# Quantum Circuits | Exact Replay and Three Verified Improvements

Observed Kaggle public score: **5.54184**. The local verified-cost projection is **5.541846**.

This notebook verifies the exact submitted circuit bytes with the unchanged official checkers, then writes the same `submission.zip`. Attach the competition data and [the public checkpoint](https://www.kaggle.com/datasets/rudrabhaskar/quantum-compilation-r6-verified-checkpoint). The checkpoint stores the archive as `scored_archive.bin` to preserve its exact ZIP bytes through dataset upload. The default run uses those two public inputs only and performs all **200,540,928 inverse-square-root shots**, **102,400 DQI shots**, and **254,464 chemistry address shots**.

The readable source below also exposes our arithmetic, compiler and chemistry transformations. Exact checkpoint replay is the default because numerical fitting, SVD and eigendecomposition can change regenerated bytes across library versions. Every optional regeneration receives its own full report; a report is never transferred to another hash.


## What comes from each source

The inverse-square-root approximation starts with [Sumit Chongder's signed-digit polynomial compiler, script 361357149](https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149). We retain its single-bit normalization, fitted quadratic approximation and coefficient lookup. Our additions are a derivative-register split, top-carry reuse and an early static-high phase checkpoint.

The DQI decoder, original C60 chemistry factorization and eight compiled ANGLE lookup circuits came from [k256.dev's originally public `claude`, version 1](https://www.kaggle.com/code/k256net/claude?scriptVersionId=361337610). That upstream page is currently unavailable. The attached checkpoint preserves the selected compiled bytes, so exact replay does not depend on that link. Their unavailable original generators are not claimed here. The DQI rewrite is inspired by [Kirill Tymchishin's exact-control absorption](https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42); our two compiler passes extend the proof to tracked zero factors and support facts.

Our chemistry changes rotate both `W` and `H2_I`, compress and align the ensemble, retain 34 physical-metric directions, tune the native identity offset, and perform ten final exact sweeps. SVD, eigendecomposition, Procrustes alignment and Jacobi descent are standard methods. The approximate compression still has to pass the original error and gap checks.


In [ ]:
import gc, hashlib, importlib, json, math, os, shutil, sys, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.dont_write_bytecode = True

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
SOURCE, SELECTED, REPORTS = WORK / 'source', WORK / 'selected', WORK / 'verification'
COMPILED = WORK / 'compiled'
for folder in [WORK, SOURCE, SELECTED, REPORTS, COMPILED]:
    folder.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
CHECKPOINT_SLUG = 'quantum-compilation-r6-verified-checkpoint'
OBSERVED_PUBLIC_SCORE = 5.54184
ARCHIVE_SHA256 = '38ccc7e14be2821062345aa7a40cdc580525a4c8eda43e9e705cece2b6897652'
ARCHIVE_BYTES = 81690206
RECEIPT_SHA256 = 'aa48f4693795128355ee92fed5e78816da8194e5bbf7db3d1bceab837ffad7a6'
MEMBER_SHA256 = {
    "invsqrt.kmb": "c1c411a41c4b502e212813ef942815b0149ef3f94f6ecaf3eea48f684e297711",
    "dqi.kmb": "be01f9ce0549f4f975b4b3031cb73984311d809350aa6246fb2ffd9540ccd09e",
    "dfthcsos/angle_k_0.kmb": "a5166e5370e3858d61ae09f220ce81c0e5a9f3322ea60372a66d0c12b87fc2de",
    "dfthcsos/angle_k_1.kmb": "b4f41fef3704b24dd2eae172e1393b84113b1ca2228a053f461cd3ca17ee76d7",
    "dfthcsos/angle_k_2.kmb": "7873d91617f7c6873eb8b1a7582222c00b19af3c19bc09f495e27973f2c4d2a6",
    "dfthcsos/angle_k_3.kmb": "caa88952f006b9bf6d7ffda4615d646d834edc487948bfc1a3651447ce09fa78",
    "dfthcsos/angle_k_4.kmb": "fcc6df7101e627227fe6cd9729693cb2905e8c8fca8bf6388c98c88f1a110cf2",
    "dfthcsos/angle_k_5.kmb": "aeca9120088611964836145cbddfce764765f65bde3065f8ad5be16af8b2b617",
    "dfthcsos/angle_k_6.kmb": "ca0f9da0b36ad94a9fe739ea080b2eba2a7cd6a6ec252f1e16d375a40f0c9c11",
    "dfthcsos/angle_k_7.kmb": "6c47f8a61113e091497572ed6d9bb2fbc93afb02100c627442efdc9912883d91",
    "dfthcsos/submission.npz": "92ee63050914826b45460227f9e204aef68dfaf9949c34bc22b860672fb37748"
}
EXPECTED_COSTS = {
    "invsqrt": 170991.03969491262,
    "dqi": 2298047031660.737,
    "chemistry": 2546570136.2646174
}
EXPECTED_CHEMISTRY_SHA256 = '8aa3639458849957bfeb51f33131a73cf08b833829f39b91149a4c5714e282b7'
EXPECTED_PROJECTION = 5.541846022700022
SOURCE_SHA256 = {
    "sumit_reference.py": "c23855304f9871fed84f82b3d7a9db52022ba904855ac3173d1f1db93016d96c",
    "private_arithmetic_r4.py": "639d573bf5e2e1f501a959eee1acf477d4567b858dcc1696ed2b72b8c908bada",
    "private_arithmetic_r5.py": "ba4420ba5348fc2ddf1f32916f144cf29f0b01cdefdab3c6ac5c635f469fc680",
    "release_arithmetic.py": "c986ad7828283c84845584e02dd12c763bfee8d0bc9312bace4f1ba4ebbd9646",
    "dqi_zero_factors.py": "ef66286511427ca524ec74302abb4cfab52bc72a8d6d3dd49e88f32c2023cf1d",
    "dqi_support_facts_r4.py": "ff4a930a34317f642ae94c0dfa5caf34949bb0f73a87daf08f570f8747057899",
    "public_dqi_release.py": "94af59d44332e6d51cf439124d2565a6b2e2382e0f4698a12b917ac30a544e30",
    "release_chemistry.py": "49f733eb04c70b9f1d1c441b718bdceb161ebeb83122e018721cb5114419d0b2"
}

def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

verifiers = list(INPUT.rglob('verify_submission_invsqrt.py'))
assert len(verifiers) == 1, 'Attach exactly one copy of the competition data.'
COMP = verifiers[0].parents[2]
sys.path[:0] = [str(SOURCE), str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as iv
from dqi import verify_submission_dqi as dv
from dfthcsos import verify as cv
WORKERS = min(4, os.cpu_count() or 1)
spec = iv.ProblemSpecInvSqrt.for_n_p(10)
assert spec.shots_per_x == 64 and spec.num_shots == 200540928
assert len(MEMBER_SHA256) == 11
assert all(not name.startswith('/') and '..' not in Path(name).parts for name in MEMBER_SHA256)
print('Official Kickmix:', km.__version__, '| CPU workers:', WORKERS)


## Validate the public checkpoint

The binary archive and its receipt must agree before its eleven whitelisted members are extracted and checked. These hashes identify the submitted solution; the reports produced later are fresh results from this run.


In [ ]:
assert CHECKPOINT_SLUG != '__PUBLIC_R6_CHECKPOINT_PENDING__', 'Set the published checkpoint dataset slug before releasing this source.'
checkpoints = []
for receipt_path in INPUT.rglob('release_receipt.json'):
    if sha256_file(receipt_path) != RECEIPT_SHA256:
        continue
    payload = receipt_path.parent / 'scored_archive.bin'
    if payload.is_file() and payload.stat().st_size == ARCHIVE_BYTES and sha256_file(payload) == ARCHIVE_SHA256:
        checkpoints.append((receipt_path, payload))
assert len(checkpoints) == 1, 'Attach exactly one public checkpoint with its unexpanded binary archive.'
receipt_path, archive_payload = checkpoints[0]
CHECKPOINT = receipt_path.parent
assert CHECKPOINT_SLUG in CHECKPOINT.parts
receipt = json.loads(receipt_path.read_text())
assert receipt['archive_sha256'] == ARCHIVE_SHA256 and receipt['archive_bytes'] == ARCHIVE_BYTES
assert receipt['submission_member_sha256'] == MEMBER_SHA256
assert receipt['receipt_version'] == 1 and receipt['archive_filename'] == 'submission.zip'
assert receipt['source_manifest_sha256'] == 'c24f78d38e17723aedfb81dc477585eaaa10e5f96dfe71eeddb370a752445aef'
assert receipt['copied_without_recompilation'] is True
with zipfile.ZipFile(archive_payload) as archive:
    assert len(archive.namelist()) == 11 and set(archive.namelist()) == set(MEMBER_SHA256)
    assert archive.testzip() is None
    for name, digest in MEMBER_SHA256.items():
        target = COMPILED / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(name) as stream:
            with target.open('wb') as destination:
                shutil.copyfileobj(stream, destination, length=8 << 20)
        assert sha256_file(target) == digest, name
assert {str(path.relative_to(COMPILED)) for path in COMPILED.rglob('*') if path.is_file()} == set(MEMBER_SHA256)
recorded = receipt['selected_full_reports']
assert set(recorded) == {'invsqrt', 'dqi', 'chemistry'}
for task, shots in [('invsqrt', 200540928), ('dqi', 102400), ('chemistry', 254464)]:
    report = recorded[task]
    assert report['status'] == 'PASS' and report['verified_shots'] == shots
    assert math.isclose(report['score'], EXPECTED_COSTS[task], rel_tol=1e-12, abs_tol=1e-4)
for task in ['invsqrt', 'dqi']:
    assert recorded[task]['circuit_sha256'] == MEMBER_SHA256[task + '.kmb']
assert recorded['dqi']['value_errors'] == recorded['dqi']['phase_errors'] == 0
assert recorded['chemistry']['submission_sha256'] == EXPECTED_CHEMISTRY_SHA256
fresh_reports = {}
print('Exact public checkpoint validated:', CHECKPOINT)
print('Archive bytes:', ARCHIVE_BYTES, '| verified members:', len(MEMBER_SHA256))


## Readable compiler and chemistry sources

These eight files contain the public reference generator and the current release implementation. No historical search grids or research manifests are needed. The default run below verifies the checkpoint; regeneration examples follow the final reports.


In [ ]:
%%writefile /kaggle/working/source/sumit_reference.py
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage:  python build_invsqrt.py OUT.kmb   (needs kickmix importable and numpy)
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 10, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay):
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); gadd(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); gadd(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: gadd(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: gadd(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out)
    print('wrote', out)

if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else 'invsqrt.kmb')


In [ ]:
%%writefile /kaggle/working/source/private_arithmetic_r4.py
"""Private derivative split and top-carry reuse over Sumit Chongder's compiler.

Reference generator credited to Sumit Chongder:
https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
scriptVersionId=361357149. Original-source license metadata is not asserted here.
The normalization, signed-digit model, fitting, and phase schedule belong to
that source. Our changes split its derivative accumulator and reuse the top
adder carry. R3 and the reference source are left unchanged.
"""
from fractions import Fraction
import hashlib
import importlib.util
import json
import math
from pathlib import Path
import time

import numpy as np

REFERENCE_SHA256 = 'c23855304f9871fed84f82b3d7a9db52022ba904855ac3173d1f1db93016d96c'


def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def load_reference(path):
    assert sha256(path) == REFERENCE_SHA256
    spec = importlib.util.spec_from_file_location('_sumit_reference_r4', path)
    reference = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(reference)
    return reference


def gadd_top_reused(b, source, target, cin=None, sub=False):
    """Gidney addition with n-2 clean carries; the last carry targets the MSB."""
    import kickmix as km
    a, t = list(source), list(target)
    n, m = len(t), len(a)
    assert n >= 1 and m <= n
    if sub:
        b.x(km.array(t))
    carries = b.alloc_qubits(n - 2) if n > 2 else km.array([])
    for i in range(n - 1):
        previous = cin if i == 0 else carries[i - 1]
        last = i == n - 2
        out = t[-1] if last else carries[i]
        if previous is None:
            if i < m:
                (b.ccx if last else b.init_and)(a[i], t[i], out)
        elif i < m:
            b.cx(previous, a[i]); b.cx(previous, t[i])
            (b.ccx if last else b.init_and)(a[i], t[i], out)
            b.cx(previous, out)
        else:
            (b.ccx if last else b.init_and)(previous, t[i], out)
    if n - 1 < m:
        b.cx(a[n - 1], t[n - 1])
    if n == 1 and cin is not None:
        b.cx(cin, t[0])
    for i in reversed(range(n - 1)):
        previous = cin if i == 0 else carries[i - 1]
        last = i == n - 2
        if previous is None:
            if i < m:
                if not last:
                    b.del_and(a[i], t[i], carries[i])
                b.cx(a[i], t[i])
        elif i < m:
            if not last:
                b.cx(previous, carries[i]); b.del_and(a[i], t[i], carries[i])
            b.cx(previous, a[i]); b.cx(a[i], t[i])
        else:
            if not last:
                b.del_and(previous, t[i], carries[i])
            b.cx(previous, t[i])
    if n > 2:
        b.free(carries)
    if sub:
        b.x(km.array(t))


def prepare_split(reference, cfg, tables):
    d = np.arange(1 << reference.DB, dtype=np.int64)
    coarse, residual = {}, {}
    bounds = {'normalized_addresses_checked': 0, 'min_residual': 1 << 30,
              'max_residual': 0, 'max_table_residual': 0, 'max_coarse': 0}
    for key, (k2, k1, _) in tables.items():
        derivative, _ = reference.pipeline(cfg, k2, k1, d)
        base = int(derivative.min()) >> 11
        low = derivative - (base << 11)
        coarse[key], residual[key] = base, k1 - (base << 11)
        assert 0 <= residual[key] < 1 << 12
        assert (low >= 0).all() and (low < 1 << 12).all()
        assert ((base + (low >> 11)) < 1 << 7).all()
        joined = (low & 2047) + ((base + (low >> 11)) << 11)
        assert np.array_equal(joined, derivative)
        bounds['normalized_addresses_checked'] += len(d)
        bounds['min_residual'] = min(bounds['min_residual'], int(low.min()))
        bounds['max_residual'] = max(bounds['max_residual'], int(low.max()))
        bounds['max_table_residual'] = max(bounds['max_table_residual'], residual[key])
        bounds['max_coarse'] = max(bounds['max_coarse'], base)
    assert cfg.w2 == 9 and len(cfg.rrow_list()) == 8
    return {'coarse': coarse, 'residual': residual}, bounds


def model_units(reference, cfg, tables, xs):
    xs = np.asarray(xs, dtype=np.int64)
    leading = 21 - np.floor(np.log2(xs)).astype(np.int64)
    normalized = xs << leading
    index = ((normalized >> 14) & 127) + ((leading & 1) << 7)
    packed = np.array([tables[e, a] for e in (0, 1) for a in range(128)], dtype=np.int64)
    k2, derivative, k0 = (packed[index, j].copy() for j in range(3))
    d = normalized & ((1 << 14) - 1)
    for i in cfg.rrow_list():
        derivative -= (2 * ((d >> i) & 1) - 1) * reference.rd(k2, cfg.sh2[i])
    value = k0.copy()
    for i in reversed(range(14)):
        value -= (2 * ((d >> i) & 1) - 1) * reference.rd(derivative, cfg.sh1[i])
    assert (value >= 0).all()
    rounded = (value >> cfg.gy) << (leading >> 1)
    return rounded.astype(np.uint64) << np.uint64(64 - (cfg.fyr + 10))


def integer_certificate(reference, cfg, tables, max_x=3133452):
    worst, margin = 0.0, 1 << 64
    for start in range(1, max_x + 1, 131072):
        xs = np.arange(start, min(start + 131072, max_x + 1), dtype=np.int64)
        actual = model_units(reference, cfg, tables, xs)
        roots = [math.isqrt((1 << 130) // int(x)) for x in xs]
        expected = np.array([((r + 1) >> 1) & ((1 << 64) - 1) for r in roots], dtype=np.uint64)
        tolerance = np.array([min(r >> 25, 1 << 63) for r in roots], dtype=np.uint64)
        error = np.abs((actual - expected).view(np.int64)).view(np.uint64)
        assert (error < tolerance).all(), 'Integer phase certificate failed.'
        worst = max(worst, float((error / tolerance).max()))
        margin = min(margin, int((tolerance - error).min()))
    return {'pass': True, 'checked_inputs': max_x, 'worst_ratio': worst,
            'min_margin_units': margin}


def build_candidate(reference, cfg, tables, layout, split):
    import kickmix as km
    assert (layout['w1'], layout['LB'], layout['S'], layout['whi']) == (18, 21, 20, 8)
    b = km.CircuitBuilder()
    x = b.create_quantum_register(22, name='x')
    flags = b.alloc_qubits(5)
    c2, residual, coarse, low, high = [b.alloc_qubits(n) for n in [9, 12, 7, 21, 8]]
    target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))
    table = [0] * 256
    for (parity, address), (k2, _, _) in tables.items():
        key = parity, address
        table[address + 128 * parity] = (layout['lo'][key] | (split['residual'][key] << 21)
            | (k2 << 33) | (layout['hi'][key] << 42) | (split['coarse'][key] << 50))
    address = km.array(list(x[14:21]) + [flags[0]])
    delta = x[:14]

    def inner(inverse):
        rows = cfg.rrow_list()
        for i in reversed(rows) if inverse else rows:
            shift = cfg.sh2[i]
            assert shift >= 0
            addend = c2[shift:]
            carry = c2[shift - 1] if shift else None
            b.cx(delta[i], residual)
            gadd_top_reused(b, addend, residual, carry, sub=inverse)
            b.cx(delta[i], residual)

    def outer(derivative, inverse):
        rows = range(14) if inverse else reversed(range(14))
        for i in rows:
            shift = cfg.sh1[i]
            addend = derivative[shift:]
            carry = derivative[shift - 1] if shift else None
            b.cx(delta[i], low)
            gadd_top_reused(b, addend, low, carry, sub=inverse)
            b.cx(delta[i], low)

    reference.normalize(b, x, flags)
    b.init_lookup(table=table, address=address, target=target)
    inner(False)
    gadd_top_reused(b, residual[11:], coarse)
    derivative = residual[:11] + coarse
    outer(derivative, False)
    ext = b.alloc_qubits(10)
    gadd_top_reused(b, low[20:], high)
    phase = km.array(list(low[cfg.gy:20]) + list(high) + list(ext))
    reference.shift_stages(b, flags, phase, 20 - cfg.gy + 8, False)
    for i, qubit in enumerate(phase):
        exponent = cfg.gy + i - cfg.fy - 10
        if exponent < 0:
            b.z_pow(qubit, Fraction(2) ** (exponent + 1))
    reference.shift_stages(b, flags, phase, 20 - cfg.gy + 8, True)
    gadd_top_reused(b, low[20:], high, sub=True)
    b.free(ext)
    outer(derivative, True)
    gadd_top_reused(b, residual[11:], coarse, sub=True)
    inner(True)
    b.del_lookup(table=table, address=address, target=target)
    b.free(target)
    reference.unnormalize(b, x, flags)
    b.free(flags)
    return b.finish_circuit()


def spot_check(circuit, reference, cfg, tables, spec):
    import kickmix as km
    from invsqrt import verify_submission_invsqrt as iv
    inputs, scratch = iv.validate_registers(circuit, spec)
    rng = np.random.default_rng(20261026)
    simulator = km.Simulator(batch_size=256, seed=20261026, debug_print=False)
    for batch in range(32):
        xs = rng.integers(1, spec.max_x + 1, 256, dtype=np.uint64)
        if batch == 0:
            edges = [1, 2, 3, spec.max_x]
            edges += [x for p in range(1, 22) for x in [2**p - 1, 2**p, 2**p + 1] if x <= spec.max_x]
            xs[:len(edges)] = edges
        simulator.clear_for_shot()
        simulator.write_across_shots(inputs, iv.to_bits(xs, spec.width)); simulator.do(circuit)
        assert np.array_equal(iv.from_bits(simulator.read_across_shots(inputs)), xs)
        assert not simulator.read_across_shots(scratch).any()
        actual = np.array([iv.half_turns_to_units(simulator.read_shot_phase(i)) for i in range(256)], dtype=np.uint64)
        assert np.array_equal(actual, model_units(reference, cfg, tables, xs))


def run_arithmetic_r4(baseline_circuit, baseline_report, reference_file, *, output, max_workers=4):
    baseline, output = Path(baseline_circuit), Path(output)
    assert baseline.resolve() != (output / 'candidate.kmb').resolve(), 'Candidate must not overwrite the verified fallback.'
    import kickmix as km
    from invsqrt import verify_submission_invsqrt as iv
    output.mkdir(parents=True, exist_ok=True)
    baseline_sha = sha256(baseline)
    assert baseline_report['status'] == 'PASS' and baseline_report['verified_shots'] == 200540928
    assert baseline_report['circuit_sha256'] == baseline_sha
    assert math.isfinite(baseline_report['score']) and baseline_report['score'] > 0
    fallback = {'improved': False, 'selected_circuit': str(baseline), 'report': dict(baseline_report)}
    audit = {'private_only': True, 'candidate': 'sumit_signed_digit_derivative_L11_top_carry_reuse',
             'reference_sha256': REFERENCE_SHA256, 'baseline_sha256': baseline_sha}
    try:
        reference = load_reference(reference_file)
        cfg = reference.Cfg(reference.F2, reference.F1, reference.FYR, reference.GY, reference.DROP2)
        tables, model_worst = reference.build_tables(cfg)
        layout = reference.layout(cfg, tables)
        split, audit['bounds'] = prepare_split(reference, cfg, tables)
        certificate = integer_certificate(reference, cfg, tables)
        audit['integer_certificate'] = certificate
        audit['reference_model_worst_relative'] = model_worst
        circuit = build_candidate(reference, cfg, tables, layout, split)
        stats = circuit.operation_statistics()
        rotations = int(stats.num_z_pow(min_clifford_hierarchy_level=4))
        epsilon = math.pi * certificate['min_margin_units'] / 2**64 / rotations
        estimated = circuit.num_qubits * (stats.num_toffoli_like +
            (stats.num_t_like + rotations * iv.rotation_t_count(epsilon)) / 4)
        audit.update(num_qubits=circuit.num_qubits, max_toffoli_count=int(stats.num_toffoli_like),
                     rotations=rotations, estimated_cost=float(estimated))
        print('PRIVATE R4 ARITHMETIC STATS', json.dumps(audit), flush=True)
        if estimated >= baseline_report['score']:
            result = {**fallback, 'audit': audit, 'reason': 'No conservative compiled-cost improvement.'}
        else:
            spec = iv.ProblemSpecInvSqrt.for_n_p(10)
            spot_check(circuit, reference, cfg, tables, spec)
            audit['spot_check_shots'] = 8192
            candidate = output / 'candidate.kmb'; circuit.to_file(candidate, format='kmb')
            digest = sha256(candidate)
            reloaded = km.Circuit.from_file(candidate, format='kmb')
            started = time.monotonic()
            report = iv.verify(reloaded, spec, max_workers=max_workers)
            report['verification_seconds'] = time.monotonic() - started
            assert report['status'] == 'PASS' and report['verified_shots'] == 200540928
            assert report['circuit_sha256'] == digest == sha256(candidate)
            assert math.isfinite(report['score']) and 0 < report['score'] < baseline_report['score']
            assert report['num_qubits'] == circuit.num_qubits and report['score'] <= estimated + 1e-6
            report['artifact_sha256'] = digest
            result = {'improved': True, 'selected_circuit': str(candidate), 'report': report, 'audit': audit}
    except Exception as error:
        result = {**fallback, 'audit': audit, 'candidate_failure': type(error).__name__ + ': ' + str(error)}
    (output / 'selected.json').write_text(json.dumps(result, indent=2))
    print('PRIVATE R4 ARITHMETIC SELECTED', json.dumps(result), flush=True)
    return result


In [ ]:
%%writefile /kaggle/working/source/private_arithmetic_r5.py
"""One private static-high phase checkpoint over the attributed R4 compiler.

Sumit Chongder's generator supplies the normalized signed-digit polynomial and
fitting: https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
scriptVersionId=361357149. R4 adds our derivative split and top-carry reuse.
This candidate phases the static high output first, erases that QROM output,
and phases the full residual later. R4 files and its arithmetic are unchanged.
"""
from fractions import Fraction
import json
import math
from pathlib import Path
import time

import numpy as np
import private_arithmetic_r4 as r4

R4_SHA256 = '639d573bf5e2e1f501a959eee1acf477d4567b858dcc1696ed2b72b8c908bada'


def build_candidate(reference, cfg, tables, layout, split):
    import kickmix as km
    assert (cfg.gy, layout['w1'], layout['LB'], layout['S'], layout['whi']) == (2, 18, 21, 20, 8)
    b = km.CircuitBuilder()
    x = b.create_quantum_register(22, name='x')
    flags = b.alloc_qubits(5)
    # These coefficients are still zero during normalization; allocate afterward.
    reference.normalize(b, x, flags)
    c2, residual, coarse, low, high = [b.alloc_qubits(n) for n in [9, 12, 7, 21, 8]]
    full_target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))
    rest_target = km.array(list(low) + list(residual) + list(c2) + list(coarse))
    full_table, rest_table, high_table = [0] * 256, [0] * 256, [0] * 256
    for (parity, address), (k2, _, _) in tables.items():
        key, index = (parity, address), address + 128 * parity
        common = layout['lo'][key] | (split['residual'][key] << 21) | (k2 << 33)
        full_table[index] = common | (layout['hi'][key] << 42) | (split['coarse'][key] << 50)
        rest_table[index] = common | (split['coarse'][key] << 42)
        high_table[index] = layout['hi'][key]
    address = km.array(list(x[14:21]) + [flags[0]])
    delta = x[:14]

    def phase_register(register, low_bit):
        ext = b.alloc_qubits(10)
        word = km.array(list(register) + list(ext))
        reference.shift_stages(b, flags, word, len(register), False)
        for i, qubit in enumerate(word):
            exponent = low_bit + i - cfg.fy - 10
            if exponent < 0:
                b.z_pow(qubit, Fraction(2) ** (exponent + 1))
        reference.shift_stages(b, flags, word, len(register), True)
        b.free(ext)

    def inner(inverse):
        rows = cfg.rrow_list()
        for i in reversed(rows) if inverse else rows:
            shift = cfg.sh2[i]
            assert shift >= 0
            carry = c2[shift - 1] if shift else None
            b.cx(delta[i], residual)
            r4.gadd_top_reused(b, c2[shift:], residual, carry, sub=inverse)
            b.cx(delta[i], residual)

    def outer(derivative, inverse):
        for i in range(14) if inverse else reversed(range(14)):
            shift = cfg.sh1[i]
            carry = derivative[shift - 1] if shift else None
            b.cx(delta[i], low)
            r4.gadd_top_reused(b, derivative[shift:], low, carry, sub=inverse)
            b.cx(delta[i], low)

    b.init_lookup(table=full_table, address=address, target=full_target)
    phase_register(high, 20)
    # High and the shift workspace have been restored. For every X-measurement
    # mask m, unlookup contributes (-1)^(m.high(address)); its phase correction
    # contributes the same sign. Their product is +1, preserving our phase.
    # Address and its parity flag stay live and unchanged until after erasure.
    b.del_lookup(table=high_table, address=address, target=high)
    b.free(high)
    inner(False)
    r4.gadd_top_reused(b, residual[11:], coarse)
    derivative = residual[:11] + coarse
    outer(derivative, False)
    # Include bit20 here. There is no output join: floor((L+2^20H)/4)
    # equals floor(L/4)+2^18H, so the two diagonal phases add exactly.
    phase_register(low[cfg.gy:], cfg.gy)
    outer(derivative, True)
    r4.gadd_top_reused(b, residual[11:], coarse, sub=True)
    inner(True)
    b.del_lookup(table=rest_table, address=address, target=rest_target)
    b.free(rest_target)
    reference.unnormalize(b, x, flags)
    b.free(flags)
    return b.finish_circuit()


def run_arithmetic_r5(baseline_circuit, baseline_report, reference_file, *, output, max_workers=4):
    baseline, output = Path(baseline_circuit), Path(output)
    assert baseline.resolve() != (output / 'candidate.kmb').resolve(), 'Candidate must not overwrite the verified fallback.'
    assert r4.sha256(r4.__file__) == R4_SHA256
    import kickmix as km
    from invsqrt import verify_submission_invsqrt as iv
    output.mkdir(parents=True, exist_ok=True)
    baseline_sha = r4.sha256(baseline)
    assert baseline_report['status'] == 'PASS' and baseline_report['verified_shots'] == 200540928
    assert baseline_report['circuit_sha256'] == baseline_sha
    assert math.isfinite(baseline_report['score']) and baseline_report['score'] > 0
    fallback = {'improved': False, 'selected_circuit': str(baseline), 'report': dict(baseline_report)}
    audit = {'private_only': True, 'candidate': 'static_high_phase_checkpoint',
             'r4_module_sha256': R4_SHA256, 'reference_sha256': r4.REFERENCE_SHA256,
             'baseline_sha256': baseline_sha}
    try:
        reference = r4.load_reference(reference_file)
        cfg = reference.Cfg(reference.F2, reference.F1, reference.FYR, reference.GY, reference.DROP2)
        tables, model_worst = reference.build_tables(cfg)
        layout = reference.layout(cfg, tables)
        split, audit['bounds'] = r4.prepare_split(reference, cfg, tables)
        certificate = r4.integer_certificate(reference, cfg, tables)
        audit['integer_certificate'] = certificate
        audit['reference_model_worst_relative'] = model_worst
        circuit = build_candidate(reference, cfg, tables, layout, split)
        stats = circuit.operation_statistics()
        rotations = int(stats.num_z_pow(min_clifford_hierarchy_level=4))
        epsilon = math.pi * certificate['min_margin_units'] / 2**64 / rotations
        estimated = circuit.num_qubits * (stats.num_toffoli_like +
            (stats.num_t_like + rotations * iv.rotation_t_count(epsilon)) / 4)
        audit.update(num_qubits=circuit.num_qubits, max_toffoli_count=int(stats.num_toffoli_like),
                     rotations=rotations, estimated_cost=float(estimated))
        print('PRIVATE R5 ARITHMETIC STATS', json.dumps(audit), flush=True)
        if not math.isfinite(estimated) or estimated >= baseline_report['score']:
            result = {**fallback, 'audit': audit, 'reason': 'No conservative compiled-cost improvement.'}
        else:
            spec = iv.ProblemSpecInvSqrt.for_n_p(10)
            r4.spot_check(circuit, reference, cfg, tables, spec)
            audit['spot_check_shots'] = 8192
            candidate = output / 'candidate.kmb'; circuit.to_file(candidate, format='kmb')
            digest = r4.sha256(candidate)
            reloaded = km.Circuit.from_file(candidate, format='kmb')
            started = time.monotonic()
            report = iv.verify(reloaded, spec, max_workers=max_workers)
            report['verification_seconds'] = time.monotonic() - started
            assert report['status'] == 'PASS' and report['verified_shots'] == 200540928
            assert report['circuit_sha256'] == digest == r4.sha256(candidate)
            assert math.isfinite(report['score']) and 0 < report['score'] < baseline_report['score']
            assert report['num_qubits'] == circuit.num_qubits and report['score'] <= estimated + 1e-6
            report['artifact_sha256'] = digest
            result = {'improved': True, 'selected_circuit': str(candidate), 'report': report, 'audit': audit}
    except Exception as error:
        result = {**fallback, 'audit': audit, 'candidate_failure': type(error).__name__ + ': ' + str(error)}
    (output / 'selected.json').write_text(json.dumps(result, indent=2))
    print('PRIVATE R5 ARITHMETIC SELECTED', json.dumps(result), flush=True)
    return result


In [ ]:
%%writefile /kaggle/working/source/release_arithmetic.py
"""Rebuild and fully verify the released 95-qubit inverse-square-root circuit.

The fitted signed-digit compiler is credited to Sumit Chongder, script361357149:
https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
Our pinned R4/R5 modules add derivative splitting, top-carry reuse, and early
static-high phase cleanup. A rebuilt artifact always receives its own full
official report; no report is transferred to a different circuit hash.
"""
import hashlib
import json
import math
from pathlib import Path
import time

import private_arithmetic_r4 as r4
import private_arithmetic_r5 as r5

R4_SHA256 = '639d573bf5e2e1f501a959eee1acf477d4567b858dcc1696ed2b72b8c908bada'
R5_SHA256 = 'ba4420ba5348fc2ddf1f32916f144cf29f0b01cdefdab3c6ac5c635f469fc680'
RECORDED_ARTIFACT_SHA256 = 'c1c411a41c4b502e212813ef942815b0149ef3f94f6ecaf3eea48f684e297711'
MAX_VERIFIED_COST = 170991.03969491262
MAX_CONSERVATIVE_COST = 172036.0832069283


def rebuild_public_arithmetic(reference_file, *, output, max_workers=4):
    """Return only a freshly verified, hash-bound artifact at the released cost."""
    assert r4.sha256(r4.__file__) == R4_SHA256
    assert r4.sha256(r5.__file__) == R5_SHA256
    output = Path(output)
    artifact = output / 'invsqrt_rebuilt.kmb'
    assert Path(reference_file).resolve() != artifact.resolve()
    import kickmix as km
    from invsqrt import verify_submission_invsqrt as iv
    output.mkdir(parents=True, exist_ok=True)
    reference = r4.load_reference(reference_file)
    cfg = reference.Cfg(reference.F2, reference.F1, reference.FYR, reference.GY, reference.DROP2)
    tables, _ = reference.build_tables(cfg)
    layout = reference.layout(cfg, tables)
    split, bounds = r4.prepare_split(reference, cfg, tables)
    certificate = r4.integer_certificate(reference, cfg, tables)
    circuit = r5.build_candidate(reference, cfg, tables, layout, split)
    stats = circuit.operation_statistics()
    rotations = int(stats.num_z_pow(min_clifford_hierarchy_level=4))
    assert circuit.num_qubits == 95 and rotations == 44
    epsilon = math.pi * certificate['min_margin_units'] / 2**64 / rotations
    conservative_cost = circuit.num_qubits * (stats.num_toffoli_like +
        (stats.num_t_like + rotations * iv.rotation_t_count(epsilon)) / 4)
    assert math.isfinite(conservative_cost) and conservative_cost <= MAX_CONSERVATIVE_COST + 1e-6
    spec = iv.ProblemSpecInvSqrt.for_n_p(10)
    assert spec.shots_per_x == 64 and spec.num_shots == 200540928
    r4.spot_check(circuit, reference, cfg, tables, spec)
    circuit.to_file(artifact, format='kmb')
    digest = r4.sha256(artifact)
    reloaded = km.Circuit.from_file(artifact, format='kmb')
    started = time.monotonic()
    # Original shots, seed derivation, batch size and verifier cost calculation.
    report = iv.verify(reloaded, spec, max_workers=max_workers)
    assert report['status'] == 'PASS' and report['verified_shots'] == 200540928
    assert report['circuit_sha256'] == digest == r4.sha256(artifact)
    assert report['num_qubits'] == 95 and report['rotation_count'] == 44
    assert math.isfinite(report['score']) and 0 < report['score'] <= MAX_VERIFIED_COST + 1e-6
    assert report['score'] <= conservative_cost + 1e-6
    report['artifact_sha256'] = digest
    report['verification_seconds'] = time.monotonic() - started
    rows = [[e, a, *map(int, tables[e, a])] for e in (0, 1) for a in range(128)]
    table_bytes = json.dumps(rows, separators=(',', ':')).encode()
    audit = {'source_url': 'https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149',
             'reference_sha256': r4.REFERENCE_SHA256, 'r4_sha256': R4_SHA256, 'r5_sha256': R5_SHA256,
             'recorded_artifact_sha256': RECORDED_ARTIFACT_SHA256,
             'byte_identical_to_recorded': digest == RECORDED_ARTIFACT_SHA256,
             'coefficient_tables_sha256': hashlib.sha256(table_bytes).hexdigest(),
             'integer_certificate': certificate, 'derivative_bounds': bounds,
             'spot_check_shots': 8192, 'conservative_cost': float(conservative_cost)}
    (output / 'invsqrt_integer_tables.json').write_bytes(table_bytes)
    result = {'selected_circuit': str(artifact), 'report': report, 'audit': audit}
    (output / 'invsqrt_rebuild_verification.json').write_text(json.dumps(result, indent=2))
    return result


In [ ]:
%%writefile /kaggle/working/source/dqi_zero_factors.py
"""Conservative semantic rewrites of official KMX text, never binary patches.

Inspiration: Kirill Tymchishin's public exact control-absorption rewrite
(https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42).
This implementation independently propagates certified zero-factor facts.
Unsupported instructions are barriers: they discard all accumulated facts.
"""
import collections
import re
import time

QUBIT = re.compile(r'q[0-9]+\Z')
ARITY = {'R': 1, 'X': 1, 'CX': 2, 'CCX': 3, 'Z': 1, 'CZ': 2, 'CCZ': 3, 'SWAP': 2, 'CSWAP': 3}


class ZeroFactors:
    """F(q) contains unchanged controls c for which c=0 implies q=0.

    Versioned references invalidate a factor whenever its control is written.
    Facts are valid only within one uninterrupted supported instruction region.
    Consequently classical condition changes and unsupported quantum operations
    cannot carry a fact into another control context.
    """
    def __init__(self):
        self.versions = collections.defaultdict(int)
        self.factors = {}
        self.constants = {}
        self.changes = collections.Counter()
        self.examples = []
        self.opcodes = collections.Counter()
        self.unknown_examples = {}

    def clear(self):
        self.versions.clear(); self.factors.clear(); self.constants.clear()

    def valid(self, factors):
        return {item for item in factors if self.versions[item[0]] == item[1]}

    def get(self, q):
        return self.valid(self.factors.get(q, set())) | {(q, self.versions[q])}

    def write(self, q, factors=(), constant=None):
        self.versions[q] += 1
        # Dropping a certified factor is safe; bound propagation memory.
        self.factors[q] = set(sorted(self.valid(factors))[:32])
        if constant is None:
            self.constants.pop(q, None)
        else:
            self.constants[q] = constant

    def product(self, controls):
        if any(self.constants.get(q) == 0 for q in controls):
            return None
        remaining = [q for q in controls if self.constants.get(q) != 1]
        changed = True
        while changed and len(remaining) > 1:
            changed = False
            for q in remaining:
                if any(other != q and (q, self.versions[q]) in self.get(other) for other in remaining):
                    remaining.remove(q); changed = True
                    break
        return remaining

    def xor_target(self, controls, target):
        if any(self.constants.get(q) == 0 for q in controls):
            return
        known = all(self.constants.get(q) == 1 for q in controls)
        old_constant = self.constants.get(target)
        if known:
            self.write(target, constant=None if old_constant is None else old_constant ^ 1)
            return
        product_factors = set().union(*(self.get(q) for q in controls))
        factors = product_factors if old_constant == 0 else self.get(target) & product_factors
        self.write(target, factors)

    def rewrite(self, line, number):
        words = line.split()
        if not words or words[0] == 'INFO_TAG':
            return line
        op, args = words[0], words[1:]
        self.opcodes[op] += 1
        if op not in ARITY or len(args) != ARITY[op] or not all(QUBIT.fullmatch(q) for q in args) or len(set(args)) != len(args):
            self.unknown_examples.setdefault(op, line.strip()[:180])
            self.clear()
            return line
        replacement = None
        reason = None
        if op == 'R':
            self.write(args[0], constant=0)
        elif op == 'X':
            q = args[0]; old = self.constants.get(q)
            self.write(q, constant=None if old is None else old ^ 1)
        elif op in {'CX', 'CCX'}:
            controls, target = args[:-1], args[-1]
            simple = self.product(controls)
            if op == 'CCX' and simple != controls:
                replacement = '' if simple is None else ('X ' + target if not simple else 'CX ' + simple[0] + ' ' + target)
                reason = 'zero or implied Toffoli control'
            self.xor_target(controls, target)
        elif op in {'Z', 'CZ', 'CCZ'}:
            simple = self.product(args)
            if op == 'CCZ' and simple != args:
                if simple is None:
                    replacement = ''
                elif not simple:
                    # Z on a certified |1> reproduces the required global phase.
                    replacement = 'Z ' + args[0]
                else:
                    replacement = ('Z' if len(simple) == 1 else 'CZ') + ' ' + ' '.join(simple)
                reason = 'zero or implied phase control'
        elif op == 'SWAP':
            a, b = args
            fa, fb = self.get(a), self.get(b)
            ca, cb = self.constants.get(a), self.constants.get(b)
            self.versions[a] += 1; self.versions[b] += 1
            self.factors[a], self.factors[b] = self.valid(fb), self.valid(fa)
            self.constants.pop(a, None); self.constants.pop(b, None)
            if cb is not None: self.constants[a] = cb
            if ca is not None: self.constants[b] = ca
        elif op == 'CSWAP':
            control, a, b = args
            ca, cb, cc = self.constants.get(a), self.constants.get(b), self.constants.get(control)
            both_masked = ((control, self.versions[control]) in self.get(a)
                           and (control, self.versions[control]) in self.get(b))
            if cc == 0 or ca == cb == 0:
                replacement = ''; reason = 'inactive Fredkin swap'
                # Exact identity: the operand facts remain unchanged.
            else:
                fa, fb = self.get(a), self.get(b)
                if cc == 1 or both_masked:
                    replacement = 'SWAP ' + a + ' ' + b; reason = 'implied Fredkin control'
                shared = fa & fb
                self.versions[a] += 1; self.versions[b] += 1
                self.factors[a] = self.valid(fb if cc == 1 or both_masked else shared)
                self.factors[b] = self.valid(fa if cc == 1 or both_masked else shared)
                self.constants.pop(a, None); self.constants.pop(b, None)
                if cc == 1 or both_masked:
                    if cb is not None: self.constants[a] = cb
                    if ca is not None: self.constants[b] = ca
                elif ca is not None and ca == cb:
                    self.constants[a] = self.constants[b] = ca
        if replacement is None:
            return line
        self.changes[op] += 1
        if len(self.examples) < 80:
            self.examples.append({'line': number, 'before': line.strip(), 'after': replacement, 'reason': reason})
        return replacement + '\n' if replacement else ''


def optimize_stream(source, destination, *, max_seconds=180, max_output_bytes=None):
    optimizer = ZeroFactors()
    started, written = time.monotonic(), 0
    for number, line in enumerate(source, 1):
        if number % 10000 == 0:
            if time.monotonic() - started > max_seconds:
                raise TimeoutError('Bounded DQI text optimization exceeded its time budget.')
            if max_output_bytes is not None and written > max_output_bytes:
                raise OSError('Bounded DQI text optimization exceeded its disk budget.')
        rewritten = optimizer.rewrite(line, number)
        destination.write(rewritten)
        written += len(rewritten)
    return {'rewrites': dict(optimizer.changes), 'rewrite_count': sum(optimizer.changes.values()),
            'examples': optimizer.examples, 'opcodes': dict(optimizer.opcodes),
            'unsupported_examples': optimizer.unknown_examples,
            'output_bytes': written, 'seconds': time.monotonic() - started}


def self_check():
    """Exhaustive basis/phase equivalence of seeded and adversarial small programs."""
    import io
    import itertools
    import random

    def evaluate(program, value, n):
        bits = [(value >> i) & 1 for i in range(n)]
        phase = 0
        for line in program.splitlines():
            words = line.split()
            if not words or words[0] in {'INFO_TAG', 'BARRIER'}: continue
            op = words[0]; q = [int(a[1:]) for a in words[1:]]
            if op == 'R': bits[q[0]] = 0
            elif op == 'X': bits[q[0]] ^= 1
            elif op == 'CX': bits[q[1]] ^= bits[q[0]]
            elif op == 'CCX': bits[q[2]] ^= bits[q[0]] & bits[q[1]]
            elif op == 'Z': phase ^= bits[q[0]]
            elif op == 'CZ': phase ^= bits[q[0]] & bits[q[1]]
            elif op == 'CCZ': phase ^= bits[q[0]] & bits[q[1]] & bits[q[2]]
            elif op == 'SWAP': bits[q[0]], bits[q[1]] = bits[q[1]], bits[q[0]]
            elif op == 'CSWAP':
                if bits[q[0]]: bits[q[1]], bits[q[2]] = bits[q[2]], bits[q[1]]
            else: raise AssertionError(op)
        return tuple(bits), phase

    programs = [
        'R q2\nCCX q0 q1 q2\nCCX q2 q0 q3\n',
        'R q2\nCCX q0 q1 q2\nX q0\nCCX q2 q0 q3\n',
        'R q2\nR q3\nCCX q0 q1 q2\nCX q2 q3\nCCZ q2 q3 q4\n',
        'R q2\nCCX q0 q1 q2\nBARRIER\nCCX q2 q0 q3\n',
        'R q2\nR q3\nCCX q0 q1 q2\nCX q2 q3\nCSWAP q0 q2 q3\n',
    ]
    rng = random.Random(20261023)
    for _ in range(220):
        lines = ['R q4', 'R q5', 'CCX q0 q1 q4', 'CX q4 q5']
        for _ in range(70):
            op = rng.choice(list(ARITY) + ['BARRIER'])
            args = [] if op == 'BARRIER' else rng.sample(range(6), ARITY[op])
            lines.append(op + ' ' + ' '.join('q' + str(i) for i in args))
        programs.append('\n'.join(lines) + '\n')
    cases, rewrites = 0, 0
    for program in programs:
        out = io.StringIO()
        audit = optimize_stream(io.StringIO(program), out)
        rewrites += audit['rewrite_count']
        for value in range(64):
            assert evaluate(program, value, 6) == evaluate(out.getvalue(), value, 6), (program, out.getvalue(), value)
            cases += 1
    assert rewrites > 100
    # A new control context is an unsupported-instruction barrier.
    opt = ZeroFactors()
    for number, line in enumerate(['R q2\n', 'CCX q0 q1 q2\n', 'PUSH_CONDITION b0\n', 'CCX q2 q0 q3\n'], 1):
        output = opt.rewrite(line, number)
    assert output == 'CCX q2 q0 q3\n'
    return {'basis_phase_cases': cases, 'programs': len(programs), 'safe_rewrites_exercised': rewrites}


if __name__ == '__main__':
    print(self_check())


In [ ]:
%%writefile /kaggle/working/source/dqi_support_facts_r4.py
"""Private R4 support-fact propagation; all control-scope boundaries stay barriers.

If q=1 implies c=1 on computational-basis support, diagonal phases and classical
bit operations preserve that implication. HMR can only shrink support on the
unmeasured qubits; invalidate the measured qubit and every reference to its old
version. Conditional HMR does not certify an unconditional reset. No branch
merging, decoder algebra, or frozen R3 module is changed.
"""

import collections
import re
import time

from dqi_zero_factors import ARITY, QUBIT, ZeroFactors, self_check as inherited_self_check

BIT = re.compile(r'b[0-9]+\Z')
CLASSICAL = {'REGISTER', 'APPEND_TO_REGISTER', 'BIT_STORE0', 'BIT_STORE1', 'BIT_INVERT'}


class SupportFacts(ZeroFactors):
    def __init__(self):
        super().__init__()
        self.preserved = collections.Counter()

    def rewrite(self, line, number):
        words = line.split()
        if not words:
            return line
        op, args = words[0], words[1:]
        if op in CLASSICAL:
            self.opcodes[op] += 1
            self.preserved[op] += 1
            return line
        if op in {'Z', 'CZ', 'CCZ'}:
            width = ARITY[op]
            qubits, suffix = args[:width], args[width:]
            valid = (len(qubits) == width and len(set(qubits)) == width
                     and all(QUBIT.fullmatch(q) for q in qubits))
            if valid and not suffix:
                self.preserved[op] += 1
                return super().rewrite(line, number)
            if valid and len(suffix) == 2 and suffix[0] == 'if' and BIT.fullmatch(suffix[1]):
                self.opcodes[op] += 1
                self.preserved[op] += 1
                return line
        if op == 'HMR':
            valid = len(args) in {2, 4} and QUBIT.fullmatch(args[0]) and BIT.fullmatch(args[1])
            conditional = len(args) == 4
            if conditional:
                valid = valid and args[2] == 'if' and BIT.fullmatch(args[3])
            if valid:
                self.opcodes[op] += 1
                self.preserved[op] += 1
                self.write(args[0], constant=None if conditional else 0)
                return line
        return super().rewrite(line, number)


def optimize_stream(source, destination, *, max_seconds=900, max_output_bytes=None):
    optimizer = SupportFacts()
    started, written = time.monotonic(), 0
    for number, line in enumerate(source, 1):
        if number % 10000 == 0:
            if time.monotonic() - started > max_seconds:
                raise TimeoutError('Bounded DQI text optimization exceeded its time budget.')
            if max_output_bytes is not None and written > max_output_bytes:
                raise OSError('Bounded DQI text optimization exceeded its disk budget.')
        rewritten = optimizer.rewrite(line, number)
        destination.write(rewritten)
        written += len(rewritten)
    return {'rewrites': dict(optimizer.changes), 'rewrite_count': sum(optimizer.changes.values()),
            'examples': optimizer.examples, 'opcodes': dict(optimizer.opcodes),
            'preserved_operation_counts': dict(optimizer.preserved),
            'unsupported_examples': optimizer.unknown_examples,
            'output_bytes': written, 'seconds': time.monotonic() - started}


def self_check():
    """Compare complete tiny Kraus operators, including every HMR outcome/phase."""
    import io
    import itertools
    import numpy as np

    inherited = inherited_self_check()
    prefix = 'R q3\nR q4\nR q5\nCCX q0 q1 q3\n'
    programs = [prefix + phase + suffix + '\nCCX q3 q0 q5\n'
                for phase in ['Z q2', 'CZ q0 q2', 'CCZ q0 q1 q2']
                for suffix in ['', ' if b0']]
    programs += [
        prefix + 'REGISTER r0 "input"\nAPPEND_TO_REGISTER q0 r0\nBIT_STORE0 b0\nBIT_STORE1 b1\nBIT_INVERT b0 if b1\nCCX q3 q0 q5\n',
        prefix + 'CCX q1 q2 q4\nHMR q4 b2\nCZ q1 q2 if b2\nCCX q3 q0 q5\n',
        prefix + 'CCX q1 q2 q4\nHMR q4 b2 if b0\nCCX q3 q0 q5\n',
        prefix + 'HMR q0 b0\nCCX q3 q1 q5\n',
        prefix + 'HMR q0 b2 if b0\nCCX q3 q0 q5\nCCX q3 q1 q5\n',
        prefix + 'CCX q3 q2 q4\nHMR q3 b2 if b0\nCCX q4 q3 q5\nCCX q4 q0 q5\n',
        prefix + 'HMR q3 b2\nCCX q3 q2 q5\n',
        prefix + 'HMR q4 b2\nHMR q0 b3 if b2\nCCX q3 q1 q5\n',
        prefix + 'PUSH_CONDITION if b0\nZ q2\nPOP_CONDITION\nCCX q3 q0 q5\n',
        prefix + 'PUSH_CONDITION if b0\nX q0\nPOP_CONDITION\nCCX q3 q0 q5\n',
        prefix + 'H q2\nCCX q3 q0 q5\n',
        'R q3\nR q4\nR q5\nPUSH_CONDITION if b0\nCCX q0 q1 q3\nZ q2 if b1\nCCX q3 q0 q5\nPOP_CONDITION\n',
    ]

    def evaluate(program, initial, initial_bits, outcomes):
        vector, bits = initial.copy(), dict(initial_bits)
        stack, outcome_index = [], 0
        indices = np.arange(len(vector))
        for line in program.splitlines():
            words = line.split()
            if not words:
                continue
            op, args = words[0], words[1:]
            if op in {'REGISTER', 'APPEND_TO_REGISTER', 'INFO_TAG'}:
                continue
            if op == 'PUSH_CONDITION':
                assert args[0] == 'if' and len(args) == 2
                stack.append(bool(bits.get(args[1], 0)))
                continue
            if op == 'POP_CONDITION':
                stack.pop()
                continue
            enabled = all(stack)
            if 'if' in args:
                split = args.index('if')
                assert len(args[split:]) == 2
                enabled = enabled and bool(bits.get(args[split + 1], 0))
                args = args[:split]
            outcome = None
            if op == 'HMR':
                outcome = outcomes[outcome_index]
                outcome_index += 1
            if not enabled:
                continue
            if op in {'BIT_STORE0', 'BIT_STORE1', 'BIT_INVERT'}:
                assert len(args) == 1
                bits[args[0]] = (bits.get(args[0], 0) ^ 1) if op == 'BIT_INVERT' else int(op == 'BIT_STORE1')
                continue
            qubits = [int(q[1:]) for q in args if q.startswith('q')]
            if op == 'R':
                assert np.allclose(vector[(indices >> qubits[0]) & 1 == 1], 0)
            elif op in {'X', 'CX', 'CCX'}:
                mask = np.ones(len(vector), dtype=bool)
                for control in qubits[:-1]:
                    mask &= ((indices >> control) & 1).astype(bool)
                permutation = indices.copy()
                permutation[mask] ^= 1 << qubits[-1]
                vector = vector[permutation]
            elif op in {'Z', 'CZ', 'CCZ'}:
                mask = np.ones(len(vector), dtype=bool)
                for control in qubits:
                    mask &= ((indices >> control) & 1).astype(bool)
                vector[mask] *= -1
            elif op == 'HMR':
                target = qubits[0]
                low = indices[((indices >> target) & 1) == 0]
                high = low | (1 << target)
                result = np.zeros_like(vector)
                result[low] = (vector[low] + (-1) ** outcome * vector[high]) / np.sqrt(2)
                vector = result
                bits[args[1]] = outcome
            elif op == 'H':
                target = qubits[0]
                low = indices[((indices >> target) & 1) == 0]
                high = low | (1 << target)
                result = vector.copy()
                result[low] = (vector[low] + vector[high]) / np.sqrt(2)
                result[high] = (vector[low] - vector[high]) / np.sqrt(2)
                vector = result
            else:
                raise AssertionError(op)
        assert not stack
        return vector, bits

    states = []
    for basis in range(8):
        state = np.zeros(64, dtype=complex)
        state[basis] = 1
        states.append(state)
    for values in [np.arange(1, 9) * (1 + 1j * np.arange(8)), np.exp(1j * np.arange(8) ** 2)]:
        state = np.zeros(64, dtype=complex)
        state[:8] = values / np.linalg.norm(values)
        states.append(state)
    cases = rewrites = 0
    for program in programs:
        output = io.StringIO()
        audit = optimize_stream(io.StringIO(program), output)
        rewritten = output.getvalue()
        rewrites += audit['rewrite_count']
        measurements = sum(line.startswith('HMR ') for line in program.splitlines())
        for initial_bits in ({'b0': a, 'b1': b} for a, b in itertools.product(range(2), repeat=2)):
            for outcomes in itertools.product(range(2), repeat=measurements):
                for initial in states:
                    before, before_bits = evaluate(program, initial, initial_bits, outcomes)
                    after, after_bits = evaluate(rewritten, initial, initial_bits, outcomes)
                    assert np.allclose(before, after, atol=1e-12, rtol=1e-12), (program, rewritten, initial_bits, outcomes)
                    assert before_bits == after_bits
                    cases += 1
    for hmr in ['HMR q3 b2 if b0', 'HMR q0 b2 if b0']:
        optimizer = SupportFacts()
        for number, line in enumerate(prefix.splitlines() + [hmr], 1):
            optimizer.rewrite(line + '\n', number)
        assert optimizer.constants.get(hmr.split()[1]) is None
    for barrier in ['PUSH_CONDITION if b0', 'POP_CONDITION', 'H q2', 'HMR q3 b2 if unknown']:
        optimizer = SupportFacts()
        for number, line in enumerate(prefix.splitlines() + [barrier, 'CCX q3 q0 q5'], 1):
            last = optimizer.rewrite(line + '\n', number)
        assert last == 'CCX q3 q0 q5\n'
    assert rewrites >= 12
    return {'inherited': inherited, 'kraus_basis_and_phase_cases': cases,
            'programs': len(programs), 'safe_rewrites_exercised': rewrites,
            'measurement_outcomes': 'every branch, including conditional HMR',
            'scope_policy': 'PUSH/POP and unsupported operations remain full barriers'}


if __name__ == '__main__':
    print(self_check())


In [ ]:
%%writefile /kaggle/working/source/public_dqi_release.py
"""Reproduce the released DQI artifact using public inputs and fresh verification.

Compiled BM2v3 dependency: k256.dev, claude v1, scriptVersionId=361337610.
https://www.kaggle.com/code/k256net/claude?scriptVersionId=361337610
Exact-control rewrite inspiration: Kirill Tymchishin's QROM/Tower42 notebook.
https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42
Our contribution is the two certified zero/support-fact compiler passes.
This module requires no private dataset, prior report, or research runner.
"""

import gc
import hashlib
import json
import math
import shutil
import time
from pathlib import Path

BASELINE_SHA256 = 'f41a2fb38f346edc1e0bf5f7b863bf747bdd5579273be94e9ca4c54f269cab8b'
STAGE1_SHA256 = 'd9d3beeb939064ed9e7fdda2bfa338d77de7361e634a3a0ae29396bb79e07c26'
FINAL_SHA256 = 'be01f9ce0549f4f975b4b3031cb73984311d809350aa6246fb2ffd9540ccd09e'
MODULE_SHA256 = {
    'dqi_zero_factors': 'ef66286511427ca524ec74302abb4cfab52bc72a8d6d3dd49e88f32c2023cf1d',
    'dqi_support_facts_r4': 'ff4a930a34317f642ae94c0dfa5caf34949bb0f73a87daf08f570f8747057899',
}


def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def compile_public_dqi(baseline_kmb, *, output, max_workers=4):
    """Return {selected_circuit, report, audit}; fail closed on any mismatch."""
    import kickmix as km
    from dqi import verify_submission_dqi as dv
    import dqi_zero_factors as zero
    import dqi_support_facts_r4 as support

    baseline, output = Path(baseline_kmb), Path(output)
    assert sha256(baseline) == BASELINE_SHA256
    for module in [zero, support]:
        assert sha256(module.__file__) == MODULE_SHA256[module.__name__]
    assert support.ZeroFactors is zero.ZeroFactors
    stages = [
        (zero, output / 'dqi_stage1.kmb', STAGE1_SHA256, 418, 584026),
        (support, output / 'dqi.kmb', FINAL_SHA256, 280, 583746),
    ]
    source_text, candidate_text = output / '.source.kmx', output / '.candidate.kmx'
    written_paths = [source_text, candidate_text, *(stage[1] for stage in stages),
                     output / 'dqi_verification.json', output / 'dqi_compiler_audit.json']
    assert baseline.resolve() not in {path.resolve() for path in written_paths}
    output.mkdir(parents=True, exist_ok=True)
    assert shutil.disk_usage(output).free >= 4 << 30, 'Need 4 GiB free for bounded official text round trips.'
    audit = {
        'source': 'k256net/claude version 1; scriptVersionId=361337610',
        'baseline_sha256': BASELINE_SHA256, 'compiler_sha256': MODULE_SHA256,
        'kickmix_version': km.__version__, 'self_check': support.self_check(), 'passes': [],
    }
    current = baseline
    try:
        for module, target, expected_hash, rewrites, expected_toffoli in stages:
            circuit = km.Circuit.from_file(str(current), format='kmb')
            circuit.to_file(str(source_text), format='kmx')
            del circuit
            gc.collect()
            assert source_text.stat().st_size <= 3 << 30
            with source_text.open('r') as source, candidate_text.open('w') as destination:
                measured = module.optimize_stream(source, destination, max_seconds=900,
                                                 max_output_bytes=3 << 30)
            source_text.unlink()
            assert measured['rewrites'] == {'CCX': rewrites}
            assert measured['rewrite_count'] == rewrites
            assert candidate_text.stat().st_size <= 3 << 30
            circuit = km.Circuit.from_file(str(candidate_text), format='kmx')
            candidate_text.unlink()
            stats = circuit.operation_statistics()
            assert circuit.num_qubits == 1818 and int(stats.num_toffoli_like) == expected_toffoli
            circuit.to_file(str(target), format='kmb')
            del circuit, stats
            gc.collect()
            assert sha256(target) == expected_hash
            audit['passes'].append({'optimizer': module.__name__, 'rewrite_count': rewrites,
                                    'circuit_sha256': expected_hash, 'max_toffoli_count': expected_toffoli,
                                    'optimization_seconds': measured['seconds']})
            current = target
            print('PUBLIC DQI COMPILER PASS', json.dumps(audit['passes'][-1]), flush=True)
        assert sha256(current) == FINAL_SHA256
        circuit = km.Circuit.from_file(str(current), format='kmb')
        started = time.monotonic()
        report = dv.verify(circuit, dv.ProblemSpecDQI.benchmark_instance(num_shots=102400),
                           max_workers=max_workers)
        report['verification_seconds'] = time.monotonic() - started
        assert report['status'] == 'PASS' and report['verified_shots'] == 102400
        assert report['value_errors'] == report['phase_errors'] == 0
        assert report['circuit_sha256'] == FINAL_SHA256 == sha256(current)
        assert math.isfinite(report['score']) and 0 < report['score'] < 2300794066593.29
        audit['final_sha256'] = FINAL_SHA256
        audit['full_verified_shots'] = report['verified_shots']
        (output / 'dqi_verification.json').write_text(json.dumps(report, indent=2))
        (output / 'dqi_compiler_audit.json').write_text(json.dumps(audit, indent=2))
        print('PUBLIC FULL DQI PASS', json.dumps(report), flush=True)
        return {'selected_circuit': str(current), 'report': report, 'audit': audit}
    finally:
        source_text.unlink(missing_ok=True)
        candidate_text.unlink(missing_ok=True)


In [ ]:
%%writefile /kaggle/working/source/release_chemistry.py
"""Public chemistry release: exact artifact replay and optional regeneration.

Compiled ANGLE circuits and original C60 factorization:
https://www.kaggle.com/code/k256net/claude?scriptVersionId=361337610
Their original generators are not supplied by that public notebook; they are
reused as attributed compiled artifacts. Preserve upstream attribution and
applicable notices when redistributing.

Our contributions here are the normalization-reducing exact rotations,
ensemble compression, physical-metric C34 projection and scalar identity fit.
SVD/eigendecomposition may differ across numerical libraries. Regeneration
therefore gets a NEW official report and is never assigned the recorded hash.
Exact replay requires the nine scored files to be attached from a PUBLIC
checkpoint, plus the original competition data. No private data is required.
"""
import hashlib
import json
import shutil
import time
from dataclasses import replace
from pathlib import Path

import numpy as np

SOURCE_URL = 'https://www.kaggle.com/code/k256net/claude?scriptVersionId=361337610'
SCORED_FAMILY_SHA256 = '8aa3639458849957bfeb51f33131a73cf08b833829f39b91149a4c5714e282b7'
SCORED_MEMBERS_SHA256 = {'angle_k_0.kmb': 'a5166e5370e3858d61ae09f220ce81c0e5a9f3322ea60372a66d0c12b87fc2de', 'angle_k_1.kmb': 'b4f41fef3704b24dd2eae172e1393b84113b1ca2228a053f461cd3ca17ee76d7', 'angle_k_2.kmb': '7873d91617f7c6873eb8b1a7582222c00b19af3c19bc09f495e27973f2c4d2a6', 'angle_k_3.kmb': 'caa88952f006b9bf6d7ffda4615d646d834edc487948bfc1a3651447ce09fa78', 'angle_k_4.kmb': 'fcc6df7101e627227fe6cd9729693cb2905e8c8fca8bf6388c98c88f1a110cf2', 'angle_k_5.kmb': 'aeca9120088611964836145cbddfce764765f65bde3065f8ad5be16af8b2b617', 'angle_k_6.kmb': 'ca0f9da0b36ad94a9fe739ea080b2eba2a7cd6a6ec252f1e16d375a40f0c9c11', 'angle_k_7.kmb': '6c47f8a61113e091497572ed6d9bb2fbc93afb02100c627442efdc9912883d91', 'submission.npz': '92ee63050914826b45460227f9e204aef68dfaf9949c34bc22b860672fb37748'}
PUBLIC_SOURCE_MEMBERS_SHA256 = {'angle_k_0.kmb': 'a5166e5370e3858d61ae09f220ce81c0e5a9f3322ea60372a66d0c12b87fc2de', 'angle_k_1.kmb': 'b4f41fef3704b24dd2eae172e1393b84113b1ca2228a053f461cd3ca17ee76d7', 'angle_k_2.kmb': '7873d91617f7c6873eb8b1a7582222c00b19af3c19bc09f495e27973f2c4d2a6', 'angle_k_3.kmb': 'caa88952f006b9bf6d7ffda4615d646d834edc487948bfc1a3651447ce09fa78', 'angle_k_4.kmb': 'fcc6df7101e627227fe6cd9729693cb2905e8c8fca8bf6388c98c88f1a110cf2', 'angle_k_5.kmb': 'aeca9120088611964836145cbddfce764765f65bde3065f8ad5be16af8b2b617', 'angle_k_6.kmb': 'ca0f9da0b36ad94a9fe739ea080b2eba2a7cd6a6ec252f1e16d375a40f0c9c11', 'angle_k_7.kmb': '6c47f8a61113e091497572ed6d9bb2fbc93afb02100c627442efdc9912883d91', 'submission.npz': '925b317df9f070c8c0be46e195f98781b259a23a2502025d41f330de4d95ad1c'}
RECORDED_FULL_REPORT = {'status': 'PASS', 'score': 2546570136.2646174, 'lambda_eff': 47.53701636715377, 'toffoli_count': 72984.0, 'num_qubits': 734, 'brot': 10, 'batch_size': 10, 'num_circuits': 8, 'reaction_depth': 88, 'num_ops': 1687180, 'verified_shots': 254464, 'submission_sha256': '8aa3639458849957bfeb51f33131a73cf08b833829f39b91149a4c5714e282b7', 'resources': {'brot': 10, 'keep_bits': 10, 'angles_toffoli_per_pass': 31792.0, 'angles_toffoli': 63584.0, 'angles_qubits': 123, 'angles_selection_qubits': 12, 'angles_persistent_qubits': 100, 'angles_temporary_qubits': 11, 'outer_state_prep_toffoli': 680, 'inner_state_prep_toffoli': 5870, 'givens_rotations_toffoli': 2850, 'outer_state_prep_persistent_qubits': 46, 'outer_state_prep_temporary_qubits': 195, 'outer_state_prep_qubits': 241, 'inner_state_prep_persistent_qubits': 44, 'inner_state_prep_temporary_qubits': 544, 'inner_state_prep_qubits': 588, 'heuristic_persistent_qubits': 90, 'heuristic_temporary_qubits': 544, 'heuristic_toffoli': 9400, 'heuristic_qubits': 634, 'persistent_qubits': 190, 'temporary_qubits': 544, 'toffoli': 72984.0, 'qubits': 734, 'angles_within_cap': True}, 'metrics': {'error_1': 0.0023992126718305005, 'error_2': 0.019276275121415883, 'Lambda': 289.99179185749193, 'E_SOS': -22144.196144201982, 'E_gap': 3.9227942019824695, 'lambda_eff': 47.53701636715377, 'gap_in_domain': True}}
COMBINED_ANGLE_TOFFOLIS = 63584

def normalization(matrix):
    return 0.25 * np.square(np.abs(matrix).sum(axis=1)).sum()

def rotate_ensemble(weights, identity, *, sweeps=5, seed=20261023):
    weights = np.asarray(weights, dtype=np.float64)
    identity = np.asarray(identity, dtype=np.float64)
    matrix = np.concatenate([weights, identity[:, None, :]], axis=1)
    gram = matrix @ matrix.swapaxes(1, 2)
    initial = normalization(matrix)
    rng = np.random.default_rng(seed)
    pairs = np.array(np.triu_indices(matrix.shape[2], 1)).T
    rows = np.arange(matrix.shape[0])
    progress = []
    for sweep in range(sweeps):
        started = time.monotonic()
        rng.shuffle(pairs)
        for first, second in pairs:
            a, b = matrix[:, :, first].copy(), matrix[:, :, second].copy()
            # Each candidate zeros a coefficient, the breakpoints of the L1 cost.
            theta = np.concatenate([np.zeros((len(a), 1)),
                                    np.arctan2(b, a) % (np.pi / 2)], axis=1)
            cosine, sine = np.cos(theta), np.sin(theta)
            plus = a[:, :, None] * cosine[:, None, :] + b[:, :, None] * sine[:, None, :]
            minus = -a[:, :, None] * sine[:, None, :] + b[:, :, None] * cosine[:, None, :]
            cost = np.abs(plus).sum(axis=1) ** 2 + np.abs(minus).sum(axis=1) ** 2
            best = np.argmin(cost, axis=1)
            c, s = cosine[rows, best], sine[rows, best]
            matrix[:, :, first] = a * c[:, None] + b * s[:, None]
            matrix[:, :, second] = -a * s[:, None] + b * c[:, None]
        value = normalization(matrix)
        progress.append({'sweep': sweep + 1, 'lambda_h2': float(value),
                         'ratio_to_initial': float(value / initial),
                         'seconds': time.monotonic() - started})
        print(json.dumps(progress[-1]), flush=True)
    assert np.allclose(matrix @ matrix.swapaxes(1, 2), gram, atol=1e-12, rtol=1e-12)
    assert normalization(matrix) <= initial * (1 + 1e-12)
    return matrix[:, :-1], matrix[:, -1], progress

def truncate_ensemble(params, count, *, sweeps=10):
    w = np.asarray(params['W'], dtype=np.float64)
    identity = np.asarray(params['H2_I'], dtype=np.float64)
    u = np.asarray(params['U'], dtype=np.float64)
    u /= np.linalg.norm(u, axis=-1, keepdims=True)
    augmented = np.concatenate([w, identity[:, None, :]], axis=1)
    _, _, vh = np.linalg.svd(augmented, full_matrices=False)
    transformed = augmented @ vh.swapaxes(1, 2)
    gram = (u @ u.swapaxes(1, 2)) ** 2
    discarded = transformed[:, :-1, count:]
    energy = np.einsum('rbc,rbd,rdc->rc', discarded, gram, discarded)
    offset = transformed[:, -1, count:] - discarded.sum(axis=1)
    h2_bound = float(energy.sum())
    h1_bound = float(np.sum(np.abs(offset) * np.sqrt(np.maximum(0, energy))))
    # Retain the old sparse basis as closely as the smaller subspace permits.
    basis = vh[:, :count].swapaxes(1, 2)
    importance = np.square(augmented).sum(axis=1) * np.square(basis).sum(axis=2)
    indices = np.argsort(-importance, axis=1)[:, :count]
    target = np.take_along_axis(augmented, indices[:, None], axis=2)
    retained = transformed[:, :, :count]
    left, _, right = np.linalg.svd(retained.swapaxes(1, 2) @ target, full_matrices=False)
    retained = retained @ (left @ right)
    w, identity, progress = rotate_ensemble(retained[:, :-1], retained[:, -1], sweeps=sweeps)
    result = {**params, 'W': w, 'H2_I': identity}
    report = {'ensemble_count': count, 'h2_error_increase_upper_bound': h2_bound,
              'h1_target_change_upper_bound': h1_bound,
              'lambda_h2': float(normalization(np.concatenate([w, identity[:, None]], axis=1))),
              'rotation_sweeps': progress}
    return result, report

def tune_identity_offset(inputs, words, edges, brot, official_metrics):
    from scipy.optimize import linear_sum_assignment
    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, reconstruct_x

    n = inputs.num_orbitals
    vectors = reconstruct_u_vectors(np.asarray(words) / (1 << (brot - 1)), edges)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    one_body = vectors[:n]
    rank, bases, _ = inputs.w.shape
    x = reconstruct_x(vectors[n:].reshape(rank, bases, n), inputs.w)
    x = np.tril(x) + np.swapaxes(np.tril(x, k=-1), -1, -2)
    hbase = (inputs.ref['h1_thc'] - 0.5 * inputs.ref['electrons'] * inputs.S
             + inputs.beta_1 * np.eye(n))
    contraction = np.einsum('rc,rcpq->pq', inputs.W, x, optimize=True)
    abs_weights = np.abs(inputs.w).sum(axis=1)
    sum_weights = inputs.w.sum(axis=1)
    offset_norm_squared = float(np.square(inputs.W).sum())

    def evaluate(scale):
        h1 = hbase - scale * contraction
        eigenvalues, eigenvectors = np.linalg.eigh(h1)
        rows, cols = linear_sum_assignment(-np.abs(one_body @ eigenvectors))
        matched = np.empty_like(eigenvalues)
        matched[rows] = eigenvalues[cols]
        reconstructed = np.einsum('j,jp,jq->pq', matched, one_body, one_body, optimize=True)
        error_1 = float(np.linalg.norm(reconstructed - h1))
        lambda_1 = float(np.abs(eigenvalues).sum())
        lambda_2 = float(0.25 * np.square(abs_weights + np.abs(scale * inputs.W + sum_weights)).sum())
        normalization = lambda_1 + lambda_2
        e_sos = float(inputs.ref['ecore'] - inputs.ref['electrons'] * inputs.beta_1
                      - (lambda_1 - eigenvalues.sum())
                      - 0.5 * scale * scale * offset_norm_squared)
        gap = float(inputs.energy_upper - e_sos)
        valid = error_1 <= 0.02 and gap > 3.914899 and normalization > 49.0 and gap < 2 * normalization
        effective = float(np.sqrt(gap * (2 * normalization - gap))) if valid else float('inf')
        return {'scale': float(scale), 'error_1': error_1, 'Lambda': normalization,
                'E_gap': gap, 'E_SOS': e_sos, 'lambda_eff': effective}

    reference = evaluate(1.0)
    for key in ['error_1', 'Lambda', 'E_gap', 'E_SOS']:
        assert np.isclose(reference[key], official_metrics[key], atol=1e-7, rtol=1e-9), key
    if official_metrics['error_2'] > 0.02:
        return None
    offsets = np.r_[np.logspace(-7, -1, 13), 0.3, 0.5]
    grid = np.unique(np.r_[1.0, 1 - offsets, 1 + offsets])
    candidates = [evaluate(t) for t in grid]
    for _ in range(2):
        index = min(range(len(candidates)), key=lambda i: candidates[i]['lambda_eff'])
        if not np.isfinite(candidates[index]['lambda_eff']):
            return None
        low, high = grid[max(0, index - 1)], grid[min(len(grid) - 1, index + 1)]
        grid = np.unique(np.r_[grid[index], np.linspace(low, high, 101)])
        candidates = [evaluate(t) for t in grid]
    best = min(candidates, key=lambda row: row['lambda_eff'])
    params = {**inputs.params, 'H2_I': sum_weights + best['scale'] * inputs.W}
    return params, best

def build_physical_c34(params, decoded_vectors):
    """Keep the top 34 physical/identity directions, then five exact sweeps."""
    w = np.asarray(params['W'], dtype=np.float64)
    identity = np.asarray(params['H2_I'], dtype=np.float64)
    rank, bases, count = w.shape
    n = params['U'].shape[-1]
    assert count == 44 and identity.shape == (rank, count)
    vectors = np.array(decoded_vectors[n:], dtype=np.float64, copy=True).reshape(rank, bases, n)
    vectors /= np.linalg.norm(vectors, axis=-1, keepdims=True)
    assert np.isfinite(vectors).all()
    offset = identity - w.sum(axis=1)
    gram = (vectors @ vectors.swapaxes(1, 2)) ** 2
    metric = w.swapaxes(1, 2) @ gram @ w + offset[:, :, None] * offset[:, None, :]
    _, eigenvectors = np.linalg.eigh(metric)
    basis, discarded = eigenvectors[:, :, -34:], eigenvectors[:, :, :-34]
    augmented = np.concatenate([w, identity[:, None, :]], axis=1)
    retained = augmented @ basis
    projected_gram = retained @ retained.swapaxes(1, 2)
    importance = np.square(augmented).sum(axis=1) * np.square(basis).sum(axis=2)
    indices = np.argsort(-importance, axis=1)[:, :34]
    target = np.take_along_axis(augmented, indices[:, None], axis=2)
    left, _, right = np.linalg.svd(retained.swapaxes(1, 2) @ target, full_matrices=False)
    retained = retained @ (left @ right)
    before = float(normalization(retained))
    new_w, new_identity, sweeps = rotate_ensemble(
        retained[:, :-1], retained[:, -1], sweeps=5, seed=20261023,
    )
    candidate = {**params, 'W': new_w, 'H2_I': new_identity}
    after = np.concatenate([new_w, new_identity[:, None, :]], axis=1)
    assert new_w.shape == (rank, bases, 34) and np.isfinite(after).all()
    assert np.allclose(after @ after.swapaxes(1, 2), projected_gram, atol=1e-12, rtol=1e-12)
    assert all(np.array_equal(candidate[key], params[key])
               for key in params if key not in {'W', 'H2_I'})
    dropped_w = w @ discarded
    dropped_offset = np.einsum('rc,rck->rk', offset, discarded)
    x = np.einsum('rbp,rbq,rbk->rkpq', vectors, vectors, dropped_w, optimize=True)
    flat = x.reshape(-1, n * n)
    delta_h2 = float(np.linalg.norm(flat @ flat.T))
    delta_h1 = float(np.linalg.norm(np.einsum('rk,rkpq->pq', dropped_offset, x)))
    offset_loss = float(np.square(dropped_offset).sum())
    report = {
        'method': 'physical-metric C34 projection', 'C': 34, 'alpha': 1.0,
        'rotation_sweeps': sweeps, 'lambda_h2_before_rotations': before,
        'lambda_h2_after_rotations': float(normalization(after)),
        'hamiltonian_delta_frobenius': delta_h2,
        'h1_target_delta_frobenius': delta_h1,
        'discarded_native_offset_norm_squared': offset_loss,
        'gap_change_absolute_upper_bound': float(2 * np.sqrt(n) * delta_h1 + 0.5 * offset_loss),
        'limits': 'Conservative bounds do not replace official error1/error2 and gap checks.',
    }
    return candidate, report

def continue_exact_c34(params):
    """One fixed ten-sweep candidate; other factorization fields stay exact."""
    w, identity = np.asarray(params['W']), np.asarray(params['H2_I'])
    assert w.shape == (65, 60, 34) and identity.shape == (65, 34)
    augmented = np.concatenate([w, identity[:, None, :]], axis=1)
    before_gram = augmented @ augmented.swapaxes(1, 2)
    before = float(normalization(augmented))
    new_w, new_identity, sweeps = rotate_ensemble(w, identity, sweeps=10, seed=20261023)
    candidate = {**params, 'W': new_w, 'H2_I': new_identity}
    after = np.concatenate([new_w, new_identity[:, None, :]], axis=1)
    after_gram = after @ after.swapaxes(1, 2)
    assert np.isfinite(after).all()
    assert np.allclose(after_gram, before_gram, atol=1e-12, rtol=1e-12)
    assert all(np.array_equal(candidate[key], params[key])
               for key in params if key not in {'W', 'H2_I'})
    value = float(normalization(after))
    assert value <= before * (1 + 1e-12)
    return candidate, {
        'method': 'ten exact C34 ensemble-rotation sweeps', 'C': 34,
        'lambda_h2_before': before, 'lambda_h2_after': value,
        'augmented_gram_max_error': float(np.max(np.abs(after_gram - before_gram))),
        'sweeps': sweeps,
    }

def _family(path):
    path = Path(path)
    return path / 'dfthcsos' if (path / 'dfthcsos').is_dir() else path


def _sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def _check_members(source, expected):
    actual = {name: _sha256(source / name) for name in expected}
    assert actual == expected, 'Public chemistry input is not the pinned family.'
    return actual


def _copy_family(source, factorization, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for name in SCORED_MEMBERS_SHA256:
        incoming = Path(factorization) if name == 'submission.npz' else source / name
        shutil.copy2(incoming, destination / name)


def verify_scored_chemistry(public_checkpoint, *, output):
    """Copy exact scored bytes and run the unchanged full official verifier."""
    from dfthcsos import verify as cv

    source, output = _family(public_checkpoint), Path(output)
    destination = output / 'dfthcsos'
    assert source.resolve() not in {output.resolve(), destination.resolve()}
    _check_members(source, SCORED_MEMBERS_SHA256)
    edges = cv.load_tree_edges(None, source)
    assert cv.submission_sha256(cv.load_circuits(source), source / 'submission.npz', edges) == SCORED_FAMILY_SHA256
    _copy_family(source, source / 'submission.npz', destination)
    _check_members(destination, SCORED_MEMBERS_SHA256)
    started = time.monotonic()
    full = cv.verify(destination, salt='public-r6-exact-replay')
    assert full['status'] == 'PASS' and full['verified_shots'] == 254464
    assert full['submission_sha256'] == SCORED_FAMILY_SHA256
    assert full['resources'] == RECORDED_FULL_REPORT['resources']
    assert np.isclose(full['score'], RECORDED_FULL_REPORT['score'], atol=1e-4, rtol=1e-10)
    assert full['resources']['angles_toffoli'] == 2 * full['resources']['angles_toffoli_per_pass']
    assert np.isclose(full['score'], cv.score_from_metrics(full['metrics'], full['resources']), rtol=1e-12)
    _check_members(destination, SCORED_MEMBERS_SHA256)
    result = {'mode': 'exact_scored_artifact_replay', 'selected_family': str(destination),
              'report': full, 'member_sha256': SCORED_MEMBERS_SHA256,
              'source_url': SOURCE_URL, 'seconds': time.monotonic() - started}
    (output / 'chemistry_release_verification.json').write_text(json.dumps(result, indent=2))
    return result


def regenerate_public_chemistry(public_c60_family, *, output):
    """Optional fixed winning-method path from public inputs; report actual bytes.

    This omits the R2 C36 benchmark branch and all rank grids. A successful
    regenerated candidate can have a different hash/cost from the scored
    checkpoint. It receives its own complete official verification.
    """
    from dfthcsos import verify as cv
    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors

    source, output = _family(public_c60_family), Path(output)
    assert source.resolve() not in {output.resolve(), (output / 'dfthcsos').resolve()}
    _check_members(source, PUBLIC_SOURCE_MEMBERS_SHA256)
    output.mkdir(parents=True, exist_ok=True)
    family, edges = cv.load_circuits(source), cv.load_tree_edges(None, source)
    inputs_base = cv.load_inputs(source / 'submission.npz')
    with np.load(source / 'submission.npz', allow_pickle=False) as archive:
        original = {key: archive[key] for key in archive.files}
    assert original['W'].shape == (65, 60, 60)
    words, brot = original['angle_words'], 10

    rotated_w, rotated_identity, exact25 = rotate_ensemble(original['W'], original['H2_I'], sweeps=25)
    c44, compression = truncate_ensemble({**original, 'W': rotated_w, 'H2_I': rotated_identity}, 44, sweeps=10)
    np.savez(output / 'intermediate_c44.npz', **c44)
    vectors = reconstruct_u_vectors(words / (1 << (brot - 1)), edges)
    c34, projection = build_physical_c34(c44, vectors)

    def screen(params, name):
        path = output / (name + '.npz')
        np.savez(path, **params)
        inputs = replace(inputs_base, factorization_file=path,
                         params={**inputs_base.params, 'W': params['W'], 'H2_I': params['H2_I']},
                         w=params['W'], W=params['H2_I'] - params['W'].sum(axis=1))
        metrics = cv.evaluate_words(inputs, words, edges, brot)
        resources = cv.resource_costs(inputs, family, brot, COMBINED_ANGLE_TOFFOLIS)
        assert resources['angles_toffoli'] == 2 * resources['angles_toffoli_per_pass'] == COMBINED_ANGLE_TOFFOLIS
        return inputs, metrics, resources, cv.score_from_metrics(metrics, resources)

    inputs, metrics, resources, score = screen(c34, 'c34_raw')
    selected, fit = c34, None
    if metrics['error_2'] <= cv.ERROR_2_THRESHOLD:
        tuned = tune_identity_offset(inputs, words, edges, brot, metrics)
        if tuned is not None:
            adjusted, diagnostics = tuned
            tuned_params = {**c34, 'H2_I': adjusted['H2_I']}
            _, tuned_metrics, tuned_resources, tuned_score = screen(tuned_params, 'c34_identity_tuned')
            assert np.isclose(tuned_metrics['error_2'], metrics['error_2'], atol=1e-12, rtol=1e-10)
            if tuned_score < score:
                selected, metrics, resources, score, fit = tuned_params, tuned_metrics, tuned_resources, tuned_score, diagnostics
    assert np.isfinite(score), cv.rejection_reason(metrics, resources)
    final, exact10 = continue_exact_c34(selected)
    _, final_metrics, final_resources, final_score = screen(final, 'c34_rotated')
    assert np.isfinite(final_score), cv.rejection_reason(final_metrics, final_resources)
    assert final_score <= score * (1 + 1e-10)
    for key in ['error_1', 'error_2', 'E_SOS', 'E_gap']:
        assert np.isclose(final_metrics[key], metrics[key], atol=1e-7, rtol=0), key
    destination = output / 'dfthcsos'
    _copy_family(source, output / 'c34_rotated.npz', destination)
    expected = cv.submission_sha256(cv.load_circuits(destination), destination / 'submission.npz', edges)
    full = cv.verify(destination, salt='public-winning-method-regeneration')
    assert full['status'] == 'PASS' and full['verified_shots'] == 254464
    assert full['submission_sha256'] == expected
    assert cv.submission_sha256(cv.load_circuits(destination), destination / 'submission.npz', edges) == expected
    assert full['resources']['angles_toffoli'] == 2 * full['resources']['angles_toffoli_per_pass'] == COMBINED_ANGLE_TOFFOLIS
    assert np.isclose(full['score'], cv.score_from_metrics(full['metrics'], full['resources']), rtol=1e-12)
    result = {'mode': 'regenerated_candidate_with_fresh_full_verification',
              'selected_family': str(destination), 'report': full,
              'matches_scored_r6_hash': expected == SCORED_FAMILY_SHA256,
              'within_recorded_cost': full['score'] <= RECORDED_FULL_REPORT['score'],
              'source_url': SOURCE_URL, 'stages': {'exact25': exact25, 'C44': compression,
                  'physical_C34': projection, 'identity_fit': fit, 'exact10': exact10}}
    (output / 'chemistry_regeneration_verification.json').write_text(json.dumps(result, indent=2))
    return result


In [ ]:
for name, digest in SOURCE_SHA256.items():
    assert sha256_file(SOURCE / name) == digest, name
    sys.modules.pop(Path(name).stem, None)
importlib.invalidate_caches()
from release_chemistry import verify_scored_chemistry
print('Pinned source files:', len(SOURCE_SHA256))


## Inverse square root: 95 qubits and 44 rotations

Centered signed-digit Horner rows use additions instead of controlled partial products. Our derivative split keeps a 12-bit residual and seven coarse bits, joining them only while the outer rows need the derivative. Reusing the top adder carry avoids an extra carry qubit.

The static output component is phased and erased before Horner. The residual retains its overflow bit, so

$$\left\lfloor\frac{L+2^{20}H}{4}\right\rfloor=\left\lfloor\frac{L}{4}\right\rfloor+2^{18}H.$$

This separates the phases exactly. Restoring the shift workspace before measurement-unlookup leaves the same address function in the erased register. Each measurement sign is cancelled by its correction, preserving the desired phase on the input and shift flags. More rotations are charged at their tighter synthesis tolerance; the measured total cost decides the result.


In [ ]:
def verify_scalar(task):
    source, target = COMPILED / (task + '.kmb'), SELECTED / (task + '.kmb')
    assert source.resolve() != target.resolve()
    shutil.copy2(source, target)
    digest = sha256_file(target)
    assert digest == MEMBER_SHA256[task + '.kmb']
    circuit = km.Circuit.from_file(str(target), format='kmb')
    started = time.monotonic()
    if task == 'invsqrt':
        report = iv.verify(circuit, spec, max_workers=WORKERS)
        shots = 200540928
        assert report['num_qubits'] == 95 and report['rotation_count'] == 44
    else:
        assert task == 'dqi'
        report = dv.verify(circuit, dv.ProblemSpecDQI.benchmark_instance(num_shots=102400), max_workers=WORKERS)
        shots = 102400
        assert report['value_errors'] == report['phase_errors'] == 0
    assert report['status'] == 'PASS' and report['verified_shots'] == shots
    assert report['circuit_sha256'] == digest == sha256_file(target)
    assert math.isfinite(report['score']) and report['score'] > 0
    assert math.isclose(report['score'], EXPECTED_COSTS[task], rel_tol=1e-12, abs_tol=1e-4)
    report['verification_seconds'] = time.monotonic() - started
    fresh_reports[task] = report
    (REPORTS / (task + '_verification.json')).write_text(json.dumps(report, indent=2))
    print('OFFICIAL FULL PASS:', task, json.dumps(report), flush=True)
    del circuit
    gc.collect()
    return report

invsqrt_report = verify_scalar('invsqrt')


## DQI: exact compiler rewrites over the complete decoder

The first pass finds certified zero-factor relationships; the second carries support facts farther through adjacent operations. A Toffoli is replaced by a CNOT only when the tracked Boolean relationship proves that the removed control is redundant. The passes remove 418 and 280 static Toffolis from the attributed decoder. They preserve its decoder iteration count, confidence charge and error distribution. We require the complete original check with zero value and phase errors.


In [ ]:
dqi_report = verify_scalar('dqi')


## Chemistry: C34 with the original ANGLE lookup circuits

The physical metric weights ensemble directions by the decoded orbital vectors and native identity offset. Truncation is approximate and must satisfy the official Hamiltonian error and gap checks. The later paired rotations of `W` and `H2_I` preserve their augmented Gram matrix while reducing the L1 normalization.

The eight upstream compiled ANGLE circuits retain their exact public bytes. Resource accounting includes both forward and reverse ANGLE passes. This replay uses all 254,464 official address shots.

Recorded full-verifier runs isolate the later chemistry changes:

| Stage | Verified cost |
|---|---:|
| C44 compression and exact rotations | 2,589,924,529.8981 |
| Physical C34 projection and identity fit | 2,547,992,414.8496 |
| Ten further exact C34 sweeps | 2,546,570,136.2646 |

The last ten sweeps reduce `Lambda` from 290.313616 to 289.991792 while preserving the Hamiltonian errors and energy gap. Their resource count stays at 72,984 Toffolis and 734 qubits. The final row is rechecked below.


In [ ]:
chemistry_result = verify_scored_chemistry(COMPILED, output=SELECTED)
chemistry_report = chemistry_result['report']
assert chemistry_report['status'] == 'PASS' and chemistry_report['verified_shots'] == 254464
assert chemistry_report['submission_sha256'] == EXPECTED_CHEMISTRY_SHA256
assert math.isclose(chemistry_report['score'], EXPECTED_COSTS['chemistry'], rel_tol=1e-10, abs_tol=1e-4)
assert chemistry_report['resources']['angles_toffoli'] == 2 * chemistry_report['resources']['angles_toffoli_per_pass']
for name, digest in MEMBER_SHA256.items():
    if name.startswith('dfthcsos/'):
        assert sha256_file(SELECTED / name) == digest, name
fresh_reports['chemistry'] = chemistry_report
shutil.move(str(SELECTED / 'chemistry_release_verification.json'), str(REPORTS / 'chemistry_verification.json'))
print('OFFICIAL FULL PASS: chemistry', json.dumps(chemistry_report), flush=True)
gc.collect()


## Verified costs and exact submission archive

The table and plots use the fresh official reports above. The geometric mean is a local cost projection; the competition score is recorded separately. The final archive is copied byte for byte from the validated checkpoint only after every selected member is bound to its fresh report.


In [ ]:
order = ['invsqrt', 'dqi', 'chemistry']
labels = ['Inverse square root', 'DQI decoder', 'Chemistry']
reference_costs = np.array([2076774.233715282, 24439473914136.668, 3355599588.3155365])
assert set(fresh_reports) == set(order)
costs = np.array([fresh_reports[task]['score'] for task in order])
assert np.isfinite(costs).all() and (costs > 0).all()
gains = reference_costs / costs
projection = float(np.prod(gains) ** (1 / 3))
assert math.isclose(projection, EXPECTED_PROJECTION, rel_tol=1e-10)
summary = pd.DataFrame({'Task': labels, 'Verified cost': costs,
    'Reference / verified': gains, 'Qubits': [fresh_reports[t]['num_qubits'] for t in order],
    'Full verified shots': [fresh_reports[t]['verified_shots'] for t in order]})
display(summary.style.format({'Verified cost': '{:,.4f}', 'Reference / verified': '{:.3f}×',
                              'Full verified shots': '{:,}'}))
print('Local verified-cost projection:', projection)
print('Observed Kaggle public score:', OBSERVED_PUBLIC_SCORE if OBSERVED_PUBLIC_SCORE is not None else 'Not recorded in this source')
summary.to_csv(WORK / 'verified_costs.csv', index=False)

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'width_ratios': [1.5, 1]})
bars = axes[0].barh(labels, 100 / gains, color=['#267e99', '#5c8b77', '#b28a43'])
for bar, gain in zip(bars, gains):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{gain:.2f}×', va='center')
axes[0].invert_yaxis(); axes[0].set(xlabel='Verified cost / organizer reference (%)', xlim=(0, 100), title='All original checks retained')
axes[1].bar(['Published source', 'This circuit'], [182600.8579943692, costs[0]], color=['#aab4bd', '#267e99'])
axes[1].set(ylabel='Inverse-square-root verified cost', title='Exact workspace and phase changes')
axes[1].ticklabel_format(axis='y', style='plain')
fig.tight_layout(); fig.savefig(WORK / 'verified_costs.png', dpi=170, bbox_inches='tight'); plt.show(); plt.close(fig)
print('Arithmetic comparison source: Sumit Chongder, script 361357149, published full-verifier report.')

chemistry_steps = pd.DataFrame({'Stage': ['C44', 'Physical C34 + fit', 'Ten exact sweeps'],
    'Verified cost': [2589924529.898053, 2547992414.8496256, costs[2]]})
fig, ax = plt.subplots(figsize=(8.3, 3.4))
ax.plot(chemistry_steps['Stage'], chemistry_steps['Verified cost'] / 1e9,
        marker='o', color='#b28a43', linewidth=2)
ax.set(ylabel='Chemistry verified cost (billions)', title='Chemistry ablation: recorded full checks, final point from this run')
ax.grid(axis='y', alpha=.2)
fig.tight_layout(); fig.savefig(WORK / 'chemistry_ablation.png', dpi=170, bbox_inches='tight'); plt.show(); plt.close(fig)
chemistry_steps.to_csv(WORK / 'chemistry_ablation.csv', index=False)

assert {str(path.relative_to(SELECTED)) for path in SELECTED.rglob('*') if path.is_file()} == set(MEMBER_SHA256)
for name, digest in MEMBER_SHA256.items():
    assert sha256_file(SELECTED / name) == digest, name
for task in ['invsqrt', 'dqi']:
    assert fresh_reports[task]['circuit_sha256'] == sha256_file(SELECTED / (task + '.kmb'))
family, edges = cv.load_circuits(SELECTED / 'dfthcsos'), cv.load_tree_edges(None, SELECTED / 'dfthcsos')
assert cv.submission_sha256(family, SELECTED / 'dfthcsos/submission.npz', edges) == fresh_reports['chemistry']['submission_sha256']
del family; gc.collect()
submission = WORK / 'submission.zip'
assert archive_payload.resolve() != submission.resolve()
shutil.copyfile(archive_payload, submission)
assert submission.stat().st_size == ARCHIVE_BYTES and sha256_file(submission) == ARCHIVE_SHA256
with zipfile.ZipFile(submission) as archive:
    assert len(archive.namelist()) == 11 and set(archive.namelist()) == set(MEMBER_SHA256)
    assert archive.testzip() is None
    for name, digest in MEMBER_SHA256.items():
        with archive.open(name) as stream:
            assert hashlib.file_digest(stream, 'sha256').hexdigest() == digest
release_manifest = {'archive_filename': 'submission.zip', 'archive_sha256': ARCHIVE_SHA256,
    'archive_bytes': ARCHIVE_BYTES, 'submission_member_sha256': MEMBER_SHA256,
    'fresh_full_reports': fresh_reports, 'source_sha256': SOURCE_SHA256,
    'checkpoint_receipt_sha256': RECEIPT_SHA256,
    'public_checkpoint_slug': CHECKPOINT_SLUG, 'local_cost_projection': projection,
    'observed_public_score': OBSERVED_PUBLIC_SCORE,
    'verification': 'Fresh unchanged official verifiers over exact serialized selected members'}
(WORK / 'verification_manifest.json').write_text(json.dumps(release_manifest, indent=2))
assert COMPILED.parent == WORK and COMPILED.name == 'compiled'
shutil.rmtree(COMPILED)
print('Exact submission.zip ready:', ARCHIVE_BYTES, 'bytes | SHA-256', ARCHIVE_SHA256)


## Optional regeneration from readable source

These helpers are included in `source/`; the checkpoint replay above does not need extra inputs. Arithmetic can regenerate directly from the embedded public reference source. DQI regeneration additionally needs a preserved copy of k256.dev's original BM2v3 decoder, pinned by the helper. Chemistry regeneration needs its preserved original C60 family and the eight ANGLE circuits. Those archived original compiled inputs are not included in the final R6 checkpoint, and their upstream page is currently unavailable. The optional helpers expose our transformations but do not reconstruct the upstream unavailable generators.

```python
from release_arithmetic import rebuild_public_arithmetic
arithmetic = rebuild_public_arithmetic(SOURCE / 'sumit_reference.py', output=WORK / 'arithmetic_regeneration', max_workers=WORKERS)

from public_dqi_release import compile_public_dqi
# decoder = compile_public_dqi(PUBLIC_BASELINE / 'dqi.kmb', output=WORK / 'dqi_regeneration', max_workers=WORKERS)

from release_chemistry import regenerate_public_chemistry
# chemistry = regenerate_public_chemistry(PUBLIC_C60_FAMILY, output=WORK / 'chemistry_regeneration')
```

Optional results have their own serialized hashes and full official reports. Floating-point regeneration may differ from the checkpoint. It does not silently replace the exact submitted archive or inherit its competition score.
